- SIRALAMA
- ERİŞİM
- ARALIKLANDIRMA
- GÖRSELLEŞTİRME

In [1]:
!pip install mesa[rec]
!pip install seaborn

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.0/197.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.8/265.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.1 MB/s eta 0:00:00


In [2]:

from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
import random

# Ajan sınıfını tanımlayalım
class MyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.active = random.choice([True, False])
        self.wealth = random.randint(0, 100)
        self.group = "A" if self.unique_id % 2 == 0 else "B"  # Çift-ID'liler "A" grubu
        self.alis_fiyati = 0
        self.alis_miktari = 0



    def say_hi(self):
        print(
            f"Merhaba! Ben {self.unique_id}, "
            f"Servetim: {self.wealth}, "
            f"Aktif mi? {self.active}, "
            f"Grubum: {self.group}"
        )


    def step(self):
        # Ajanın her adımda yapacağı işlemler
        print(f"Ajan ID: {self.unique_id}- Servet: {self.wealth}")
        self.say_hi()

        self.alis_fiyati = random.randint(10, 100)
        self.alis_miktari = random.randint(1, 10)
        self.model.agent_list_al.append(self)


        self.satis_fiyati = random.randint(10, 100)
        self.satis_miktari = random.randint(1, 10)
        self.model.agent_list_sat.append(self)

        self.model.rich_agents.append(self)


# Model sınıfını tanımlayalım
class MyModel(mesa.Model):
    def __init__(self, N, width, height):
        super().__init__()
        self.num_agents = N
        self.grid = MultiGrid(width, height, True)


        self.agent_list_al = []  # Ajanları saklamak için boş liste
        self.agent_list_sat = []  # Ajanları saklamak için boş liste

        self.rich_agents = []


        # Ajanları oluşturalım
        for i in range(self.num_agents):
            agent = MyAgent(self)
            self.agents.add(agent)

            # Ajanları rastgele bir hücreye yerleştirelim
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))

        # Veri toplamak için DataCollector kullanabiliriz (opsiyonel)
        self.datacollector = DataCollector()



    # Modelde filtreleme:
    def get_group_a(self):
        return self.agents.select(lambda agent: agent.group == "A")


    # Dinamik Ajan Ekleyip Çıkarma

    def dynamic_agents(self):
        # diğer piyasalara ilişkin kötü haberler arttığında ve mevcut piyasaya ilşkin iyi haberler arttığında, piyasaya ajan girişi diğer zamanlardaki
        # girişlerden daha çok olacaktır. Bu nedenle piyasaya gelen haberlere göre ajan girişi olacak
        if random.random() > 0.5: #iyi haber geldi, 100 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.25 * len(self.agents))):   # yeni giriş yapan ajanların sayısını, mevcut ajan sayısının belli bir oranı olarak ayarlanabilir.
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan SAYISI: {(int(0.25 * len(self.agents)))}")
        else:                    # iyi haber yok, 10 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.10 * len(self.agents))):
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan sayısİ: {(int(0.10 * len(self.agents)))}")

        print(f"Güncel Ajan SAyısı: {len(self.agents)}")

        """
        # 5 ID'li ajanı sil
        agent_to_remove = next(a for a in self.agents if a.unique_id == 5)
        self.agents.remove(agent_to_remove)
        """




    def step(self):
        # Modelin her adımda yapacağı işlemler
        self.agents.do("step")
        self.dynamic_agents()

        self.datacollector.collect(self)

# Modeli oluşturalım (100 ajan, 10x10 grid)
model = MyModel(10, 10, 10)

# Tüm ajanları AgentSet olarak almak için:
all_agents = model.agents

# AgentSet'i kontrol edelim
print(f"Toplam ajan sayısı: {len(all_agents)}")
print(f"İlk ajanın ID'si: {all_agents[0].unique_id}")

# Modeli birkaç adım çalıştıralım
for i in range(10):
    print(f"------Adım {i+1}--------")
    model.step()

Toplam ajan sayısı: 10
İlk ajanın ID'si: 1
------Adım 1--------
Ajan ID: 1- Servet: 89
Merhaba! Ben 1, Servetim: 89, Aktif mi? True, Grubum: B
Ajan ID: 2- Servet: 36
Merhaba! Ben 2, Servetim: 36, Aktif mi? False, Grubum: A
Ajan ID: 3- Servet: 95
Merhaba! Ben 3, Servetim: 95, Aktif mi? True, Grubum: B
Ajan ID: 4- Servet: 88
Merhaba! Ben 4, Servetim: 88, Aktif mi? True, Grubum: A
Ajan ID: 5- Servet: 70
Merhaba! Ben 5, Servetim: 70, Aktif mi? True, Grubum: B
Ajan ID: 6- Servet: 11
Merhaba! Ben 6, Servetim: 11, Aktif mi? True, Grubum: A
Ajan ID: 7- Servet: 4
Merhaba! Ben 7, Servetim: 4, Aktif mi? False, Grubum: B
Ajan ID: 8- Servet: 68
Merhaba! Ben 8, Servetim: 68, Aktif mi? True, Grubum: A
Ajan ID: 9- Servet: 10
Merhaba! Ben 9, Servetim: 10, Aktif mi? False, Grubum: B
Ajan ID: 10- Servet: 43
Merhaba! Ben 10, Servetim: 43, Aktif mi? True, Grubum: A
Yeni eklenen ajan sayısİ: 1
Güncel Ajan SAyısı: 11
------Adım 2--------
Ajan ID: 1- Servet: 89
Merhaba! Ben 1, Servetim: 89, Aktif mi? True, Gr

In [3]:


        # Ağırlıklı ortalama hesapla
        alici_toplam_deger = sum(i.alis_fiyati * i.alis_miktari for i in model.agent_list_al)
        alici_toplam_miktar = sum(i.alis_miktari for i in model.agent_list_al)
        alici_agirlikli_ortalama = alici_toplam_deger / alici_toplam_miktar if alici_toplam_miktar != 0 else 0

        satici_toplam_deger = sum(j.satis_fiyati * j.satis_miktari for j in model.agent_list_sat)
        satici_toplam_miktar = sum(j.satis_miktari for j in model.agent_list_sat)
        satici_agirlikli_ortalama = satici_toplam_deger / satici_toplam_miktar if satici_toplam_miktar != 0 else 0

        agirlikli_ortalama = (alici_agirlikli_ortalama + satici_agirlikli_ortalama) / 2
        print("\nAğırlıklı Ortalama Fiyat:", agirlikli_ortalama)




Ağırlıklı Ortalama Fiyat: 56.35505372327569


In [4]:

# Tüm Ajanların Miktarlarını Liste Olarak Alma

alis_miktarlar = [agent.alis_miktari for agent in model.agent_list_al]
satis_miktarlar = [agent.satis_miktari for agent in model.agent_list_sat]
print("Alış miktarlar:", alis_miktarlar)
print("Satış miktarlar:", satis_miktarlar)




Alış miktarlar: [6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 8, 5, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 8, 5, 10, 8, 5, 4, 9, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 8, 5, 10, 8, 5, 4, 9, 7, 9, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 8, 5, 10, 8, 5, 4, 9, 7, 9, 4, 1, 6, 2, 2, 10, 7, 10, 3, 7, 6, 7, 6, 1, 3, 7, 10, 8, 10, 4, 6, 3, 8, 5, 10, 8, 5, 4, 9, 7, 9, 4, 1, 5, 10, 1]
Satış miktarlar: [10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 1, 10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 1, 4, 5, 10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 1, 4, 5, 4, 6, 9, 10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 1, 4, 5, 4, 6, 9, 4, 3, 9, 4, 10, 4, 4, 1, 8, 3, 2, 6, 2, 7, 1, 4, 5, 4, 6, 9, 4, 3, 9, 4, 3, 9, 10, 4,

In [5]:
# Belirli Bir Ajanın Miktarını Alma


# Ajanlara erişim
ilk_ajan_miktar = model.agent_list_al[0].alis_miktari
print("ilk sıradaki ajan miktarı:", ilk_ajan_miktar)

# 3. sıradaki ajanın miktarı
ucuncu_ajan_miktar = model.agent_list_al[2].alis_miktari
print("ucuncu sıradaki ajan miktarı:", ucuncu_ajan_miktar)

ilk sıradaki ajan miktarı: 6
ucuncu sıradaki ajan miktarı: 2


In [6]:
#  Miktarları ve Diğer Bilgileri DataFrame'e Dönüştürme (Pandas ile)


import pandas as pd

data = {
    'alici-ID': [agent.unique_id for agent in model.agent_list_al],
    'alici-Fiyat': [agent.alis_fiyati for agent in model.agent_list_al],
    'alici-Miktar': [agent.alis_miktari for agent in model.agent_list_al],
    'satici-ID': [agent.unique_id for agent in model.agent_list_sat],
    'satici-Fiyat': [agent.satis_fiyati for agent in model.agent_list_sat],
    'satici-Miktar': [agent.satis_miktari for agent in model.agent_list_sat]
}

df = pd.DataFrame(data)
print(df)

     alici-ID  alici-Fiyat  alici-Miktar  satici-ID  satici-Fiyat  \
0           1           76             6          1            15   
1           2           42             2          2            27   
2           3           50             2          3            62   
3           4           24            10          4            85   
4           5           23             7          5            90   
..        ...          ...           ...        ...           ...   
208        30           34             4         30            96   
209        31           14             1         31            25   
210        32           10             5         32            97   
211        33           22            10         33            14   
212        34           14             1         34            70   

     satici-Miktar  
0               10  
1                4  
2                4  
3                1  
4                8  
..             ...  
208              9  
209

In [7]:
# Belirli Bir Miktar Aralığındaki Ajanları Bulma

min_miktar = 4
max_miktar = 7

filtreli_ajanlar = [agent for agent in model.agent_list_al
                   if min_miktar <= agent.alis_miktari <= max_miktar]

print(f"Miktarı {min_miktar} ile {max_miktar} arasındakiler:")
for agent in filtreli_ajanlar:
    print(f"{agent.unique_id}: {agent.alis_miktari}")


Miktarı 4 ile 7 arasındakiler:
1: 6
5: 7
8: 7
9: 6
10: 7
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
22: 5
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
22: 5
25: 5
26: 4
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
22: 5
25: 5
26: 4
28: 7
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
22: 5
25: 5
26: 4
28: 7
30: 4
1: 6
5: 7
8: 7
9: 6
10: 7
11: 6
14: 7
18: 4
19: 6
22: 5
25: 5
26: 4
28: 7
30: 4
32: 5


In [8]:
# miktara göre sıralama
miktara_gore_sirali = sorted(model.agent_list_al, key=lambda x: x.alis_miktari, reverse=True)

for agent in miktara_gore_sirali:
    print(f"{agent.unique_id}: {agent.alis_miktari}")


4: 10
6: 10
4: 10
6: 10
4: 10
6: 10
4: 10
6: 10
15: 10
4: 10
6: 10
15: 10
17: 10
4: 10
6: 10
15: 10
17: 10
4: 10
6: 10
15: 10
17: 10
23: 10
4: 10
6: 10
15: 10
17: 10
23: 10
4: 10
6: 10
15: 10
17: 10
23: 10
4: 10
6: 10
15: 10
17: 10
23: 10
33: 10
27: 9
27: 9
29: 9
27: 9
29: 9
27: 9
29: 9
16: 8
16: 8
16: 8
21: 8
16: 8
21: 8
24: 8
16: 8
21: 8
24: 8
16: 8
21: 8
24: 8
16: 8
21: 8
24: 8
5: 7
8: 7
10: 7
5: 7
8: 7
10: 7
5: 7
8: 7
10: 7
5: 7
8: 7
10: 7
14: 7
5: 7
8: 7
10: 7
14: 7
5: 7
8: 7
10: 7
14: 7
5: 7
8: 7
10: 7
14: 7
5: 7
8: 7
10: 7
14: 7
28: 7
5: 7
8: 7
10: 7
14: 7
28: 7
5: 7
8: 7
10: 7
14: 7
28: 7
1: 6
9: 6
1: 6
9: 6
11: 6
1: 6
9: 6
11: 6
1: 6
9: 6
11: 6
1: 6
9: 6
11: 6
19: 6
1: 6
9: 6
11: 6
19: 6
1: 6
9: 6
11: 6
19: 6
1: 6
9: 6
11: 6
19: 6
1: 6
9: 6
11: 6
19: 6
1: 6
9: 6
11: 6
19: 6
22: 5
22: 5
25: 5
22: 5
25: 5
22: 5
25: 5
22: 5
25: 5
32: 5
18: 4
18: 4
18: 4
26: 4
18: 4
26: 4
18: 4
26: 4
30: 4
18: 4
26: 4
30: 4
7: 3
7: 3
7: 3
13: 3
7: 3
13: 3
7: 3
13: 3
20: 3
7: 3
13: 3
20: 3
7: 3
13:

In [9]:
ilk_bes_ajan = model.agent_list_al[:5]
en_yuksek_miktarlar = [agent.alis_miktari for agent in ilk_bes_ajan]

print("En yüksek miktarları alan ilk 5 ajanın miktarları:", en_yuksek_miktarlar)



En yüksek miktarları alan ilk 5 ajanın miktarları: [6, 2, 2, 10, 7]


In [ ]:
# Temel Erişim Yöntemleri

# Orijinal listeyi koruyarak kopyalama
orijinal_liste = self.agent_list_al.copy()

# Önceliğe göre sıralama
self.agent_list_al.sort(key=lambda x: x.alici_onceligi, reverse=True)

# Sıralanmış liste üzerinde miktarlara erişim
for i, agent in enumerate(self.agent_list_al):
    print(f"Sıra {i+1}: ID {agent.unique_id} - Miktar: {agent.alis_miktari} - Öncelik: {agent.alici_onceligi:.2f}")

In [ ]:
# Miktarları Liste Olarak Alma


sirali_miktarlar = [agent.alis_miktari for agent in self.agent_list_al]
print("Sıralı miktarlar:", sirali_miktarlar)

In [ ]:
# Belirli Koşullara Göre Filtreleme

# 100'den fazla miktarı olan alıcılar
buyuk_miktarlar = [agent for agent in self.agent_list_al if agent.alis_miktari > 100]
print("100+ miktarlı alıcılar:", len(buyuk_miktarlar))

# Önceliği 50'den yüksek ve miktarı 75'ten fazla olanlar
ozel_kosul = [agent for agent in self.agent_list_al
              if agent.alici_onceligi > 50 and agent.alis_miktari > 75]


In [ ]:
alis_miktarlari = [agent.alis_miktari for agent in self.agent_list_al]
print("Sıralı alış miktarları:", alis_miktarlari)

In [ ]:
# Belirli Bir Sıradaki Ajanın Miktarı


# İlk 3 ajanın miktarları
ilk_uc_miktar = [self.agent_list_al[i].alis_miktari for i in range(3)]
print("İlk 3 miktar:", ilk_uc_miktar)

# 5. sıradaki ajanın miktarı
besinci_ajan_miktar = self.agent_list_al[4].alis_miktari

In [ ]:
# Orijinal Liste ile Karşılaştırma


print("\nOrijinal Sıra vs Sıralanmış Sıra:")
for orig, sirali in zip(orijinal_liste[:5], self.agent_list_al[:5]):
    print(f"Orijinal: {orig.alis_miktari} (Öncelik:{orig.alici_onceligi:.1f}) | Sıralı: {sirali.alis_miktari} (Öncelik:{sirali.alici_onceligi:.1f})")

In [ ]:
# Miktar Dağılımını Görselleştirme (Matplotlib ile)

import matplotlib.pyplot as plt

miktarlar = [agent.alis_miktari for agent in self.agent_list_al]
oneriler = [agent.alici_onceligi for agent in self.agent_list_al]

plt.figure(figsize=(10, 5))
plt.bar(range(len(miktarlar)), miktarlar)
plt.xlabel('Sıralı Ajan Indexi')
plt.ylabel('Alış Miktarı')
plt.title('Öncelik Sırasına Göre Alış Miktarları')
plt.show()

In [ ]:
# Formatlı Yazdırma (Her Eleman Ayrı Satırda)


class MyModel(mesa.Model):
    def __init__(self, N):
        self.agent_list = [MyAgent(i, self) for i in range(N)]

    def step(self):
        print("\nTüm Agent Listeleri:")
        for agent in self.agent_list:
            print(f"Agent {agent.unique_id}:")
            for item in agent.my_list:  # Liste elemanlarını tek tek yazdır
                print(f"  - {item}")

In [ ]:
# DataCollector ile Yazdırma (Model Seviyesinde)

class MyModel(mesa.Model):
    def __init__(self, N):
        self.schedule = mesa.time.RandomActivation(self)
        self.datacollector = mesa.DataCollector(
            agent_reporters={"Liste": lambda a: a.my_list}
        )

        for i in range(N):
            agent = MyAgent(i, self)
            self.schedule.add(agent)

    def step(self):
        self.schedule.step()
        self.datacollector.collect(self)
        # Tüm listeleri toplu yazdır
        print(self.datacollector.get_agent_vars_dataframe())

In [ ]:
# from tabulate import tabulate  # pip install tabulate

class MyAgent(mesa.Agent):
    def print_list(self):
        table = []
        for idx, item in enumerate(self.my_list, start=1):
            table.append([idx, item, item**2])  # Örnek hesaplama

        print(tabulate(table,
                      headers=["Sıra", "Değer", "Karesi"],
                      tablefmt="grid"))

from tabulate import tabulate  # pip install tabulate

class MyAgent(mesa.Agent):
    def print_list(self):
        table = []
        for idx, item in enumerate(self.my_list, start=1):
            table.append([idx, item, item**2])  # Örnek hesaplama

        print(tabulate(table,
                      headers=["Sıra", "Değer", "Karesi"],
                      tablefmt="grid"))

In [ ]:
# BatchRunner ile Toplu Yazdırma

results = mesa.batch_run(
    MyModel,
    parameters={"N": range(1, 3)},
    max_steps=5
)

# Sonuçları pandas DataFrame olarak yazdır
import pandas as pd
print(pd.DataFrame(results).head())

In [ ]:
# Detaylı Yazdırma (Her Agent'ın Özellikleriyle)

for agent in self.agent_list_al:
    print(f"Agent ID: {agent.unique_id}")
    print(f"  - Alış Fiyatı: {agent.alis_fiyati}")
    print(f"  - Alış Miktarı: {agent.alis_miktari}")
    print(f"  - Öncelik: {agent.alici_onceligi}")
    print("-" * 30)

In [ ]:
# Tablo Formatında Yazdırma (Pandas ile)

import pandas as pd

# DataFrame oluşturma
data = {
    'ID': [agent.unique_id for agent in self.agent_list_al],
    'Fiyat': [agent.alis_fiyati for agent in self.agent_list_al],
    'Miktar': [agent.alis_miktari for agent in self.agent_list_al],
    'Öncelik': [agent.alici_onceligi for agent in self.agent_list_al]
}

df = pd.DataFrame(data)
print(df)

In [ ]:
# Koşullu Yazdırma (Belirli Özelliklere Göre)

print("100'den Fazla Miktarı Olanlar:")
for agent in self.agent_list_al:
    if agent.alis_miktari > 100:
        print(f"ID: {agent.unique_id}, Miktar: {agent.alis_miktari}")

In [ ]:
# Sıralı Yazdırma (Önceliğe Göre)

# Önce sırala
self.agent_list_al.sort(key=lambda x: x.alici_onceligi, reverse=True)

# Sonra yazdır
for idx, agent in enumerate(self.agent_list_al, 1):
    print(f"{idx}. Sıra - ID: {agent.unique_id}, Öncelik: {agent.alici_onceligi:.2f}")

In [ ]:
# Kısa Özet Yazdırma

print(f"Toplam {len(self.agent_list_al)} alıcı agent var")
print("İlk 5 agent:")
for agent in self.agent_list_al[:5]:
    print(f"  {agent.unique_id}: {agent.alis_miktari} adet")

In [ ]:
# DataCollector ile Raporlama (Tavsiye Edilen)

class MyModel(mesa.Model):
    def __init__(self, N):
        self.datacollector = mesa.DataCollector(
            agent_reporters={
                "Alis_Fiyati": lambda a: a.alis_fiyati,
                "Alis_Miktari": lambda a: a.alis_miktari,
                "Oncelik": lambda a: a.alici_onceligi
            }
        )

    def step(self):
        self.datacollector.collect(self)
        if self.schedule.steps % 10 == 0:  # Her 10 adımda bir yazdır
            df = self.datacollector.get_agent_vars_dataframe()
            print(df.tail(10))  # Son 10 kaydı göster

In [ ]:
# Agent Özel Yazdırma Metodu


class MyAgent(mesa.Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.alis_fiyati = random.randint(90, 110)
        self.alis_miktari = random.randint(1, 100)
        self.alici_onceligi = self.alis_fiyati * 0.7 + self.alis_miktari * 0.3

    def print_details(self):
        return f"Agent {self.unique_id}: Fiyat={self.alis_fiyati}, Miktar={self.alis_miktari}, Öncelik={self.alici_onceligi:.2f}"

# Model içinde kullanım
for agent in self.agent_list_al:
    print(agent.print_details())

In [ ]:
# Batch Run için Özel Raporlama


def agent_report(model):
    return {
        "toplam_agent": len(model.agent_list_al),
        "ortalama_fiyat": sum(a.alis_fiyati for a in model.agent_list_al)/len(model.agent_list_al)
    }

results = mesa.batch_run(
    MyModel,
    parameters={"N": range(10, 50, 10)},
    max_steps=5,
    data_collection_period=1,
    model_reporters={"Rapor": agent_report}
)

pd.DataFrame(results).to_csv("sonuclar.csv")

In [ ]:
# Gelişmiş Görselleştirme

import matplotlib.pyplot as plt

class MyModel(mesa.Model):
    def visualize_agents(self):
        fiyatlar = [a.alis_fiyati for a in self.agent_list_al]
        miktarlar = [a.alis_miktari for a in self.agent_list_al]

        plt.figure(figsize=(10,5))
        plt.scatter(fiyatlar, miktarlar, c='green', alpha=0.6)
        plt.title("Alıcı Agent Dağılımı")
        plt.xlabel("Alış Fiyatı")
        plt.ylabel("Alış Miktarı")
        plt.grid(True)
        plt.show()

In [ ]:
# Mesa'nın visualization.py modülünden yararlan

"""
Önemli Notlar:
Büyük ölçekli simülasyonlarda sık yazdırma işlemleri performansı düşürebilir

DataCollector kullanımı veri analizi için en sağlıklı yöntemdir

Jupyter Notebook kullanıyorsanız IPython.display modülünden yararlanabilirsiniz
"""

from mesa.visualization.ModularVisualization import VisualizationElement

class AgentChart(VisualizationElement):
    def render(self, model):
        agent_data = [{
            "id": agent.unique_id,
            "fiyat": agent.alis_fiyati,
            "miktar": agent.alis_miktari
        } for agent in model.agent_list_al]

        return agent_data

In [ ]:
"""
self.agent_list_al = [] içerisinde yer alan aliş_miktari değerlerini
normal_ornekler = np.random.normal(loc=piyasa_fiyati, scale=standart_sapma, size=ornek_sayisi) içerisinde
yer alan değerlerle çarpmak isteyim bunu bir değişkende saklamak istesem nasıl yapabilirim
"""

# 1. Yöntem: Temel Yöntem (List Comprehension)

import numpy as np

# Normal dağılımdan örnekler
piyasa_fiyati = 100
standart_sapma = 15
ornek_sayisi = len(self.agent_list_al)
normal_ornekler = np.random.normal(loc=piyasa_fiyati, scale=standart_sapma, size=ornek_sayisi)

# Çarpım sonuçlarını saklama
carpim_sonuclari = [agent.alis_miktari * normal_ornekler[i]
                   for i, agent in enumerate(self.agent_list_al)]

print("Çarpım Sonuçları:", carpim_sonuclari)

# 2. yöntem: NumPy Optimizasyonu (Daha Hızlı)

# Alış miktarlarını numpy array'e çevirme
alis_miktarlari = np.array([agent.alis_miktari for agent in self.agent_list_al])

# Vektörel çarpım
carpim_sonuclari = alis_miktarlari * normal_ornekler

# Agent'lara sonuçları kaydetme (isteğe bağlı)
for i, agent in enumerate(self.agent_list_al):
    agent.carpim_sonucu = carpim_sonuclari[i]



# 3 . yöntem: Pandas DataFrame ile (Analiz için İdeal)


import pandas as pd

# DataFrame oluşturma
df = pd.DataFrame({
    'Agent_ID': [agent.unique_id for agent in self.agent_list_al],
    'Alis_Miktari': [agent.alis_miktari for agent in self.agent_list_al],
    'Normal_Deger': normal_ornekler
})

# Çarpım sütunu ekleme
df['Carpim_Sonucu'] = df['Alis_Miktari'] * df['Normal_Deger']

# Sonuçları göster
print(df.head())

# CSV'ye kaydetmek için:
# df.to_csv('carpim_sonuclari.csv', index=False)


"""
Önemli Notlar:
Boyut Uyumu: normal_ornekler boyutu ile self.agent_list_al boyutu aynı olmalı

Veri Tipi: alis_miktari değerlerinin numeric olduğundan emin olun

Performans: Büyük listeler için NumPy vektörel işlemler kullanın
"""


# Alternatif (Mesa'ya Entegre):

class MyAgent(mesa.Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.carpim_sonucu = None  # Sonucu saklamak için

class MyModel(mesa.Model):
    def calculate_products(self):
        normal_vals = np.random.normal(loc=100, scale=15, size=len(self.agent_list_al))
        for i, agent in enumerate(self.agent_list_al):
            agent.carpim_sonucu = agent.alis_miktari * normal_vals[i]
            print(f"Agent {agent.unique_id}: {agent.alis_miktari} x {normal_vals[i]:.2f} = {agent.carpim_sonucu:.2f}")


"""
Bu yöntemlerden ihtiyacınıza en uygun olanını seçebilirsiniz. Veri analizi yapacaksanız Pandas,
performans için NumPy, kalıcı saklama için agent attribute'u kullanabilirsiniz.
"""
















